# 3D Flow Visualization: Backward-Facing Step

This notebook visualizes 3D computational fluid dynamics (CFD) simulation data from a **backward-facing step** flow using [PyVista](https://docs.pyvista.org/). The visualization includes:

- **Q-criterion isosurfaces** for vortex identification
- **Recirculation bubble** visualization (regions of reversed flow)
- **Streamlines** to trace fluid particle paths

## The Backward-Facing Step Problem

The backward-facing step is a canonical benchmark problem in fluid mechanics. Flow enters a channel and encounters a sudden expansion (step), creating complex flow phenomena:

```
                    ──────────────────────────────────────►
    Inlet           │                                      
    ────►    ═══════╝                                      
             ▓▓▓▓▓▓▓ ← Step (height h)                     
                    │←── Recirculation Zone ──→│           
                    ▼                                      
```

### Key Flow Features

1. **Flow separation** at the step edge
2. **Recirculation zone** immediately downstream of the step (reversed flow, $u < 0$)
3. **Reattachment point** where the separated shear layer reattaches to the wall
4. **Vortical structures** in the shear layer

### Domain Parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| $x$ | $[-2.42, 14.54]$ | Streamwise direction (step at $x=0$) |
| $y$ | $[0, 1.0]$ | Wall-normal direction |
| $z$ | $[0, 3.88]$ | Spanwise direction |
| $h$ | $0.4845$ | Step height |
| Grid | $301 \times 76 \times 76$ | Computational mesh points |

---

## Core Functions and Theory

### Data Loading and Grid Creation

The simulation data is stored in NumPy's `.npz` format with the velocity field $\mathbf{u} = (u, v, w)$ and grid coordinates $(x, y, z)$.

**Array Ordering Convention:**

PyVista expects data in $(n_x, n_y, n_z)$ ordering with Fortran-style (column-major) flattening. The CFD solver outputs data in $(n_z, n_y, n_x)$ ordering, so we transpose:

```python
u_transposed = u.transpose(2, 1, 0).flatten(order='F')
```

### The Q-Criterion for Vortex Identification

The **Q-criterion** is a widely-used method for identifying vortical structures in fluid flows. It is based on the decomposition of the velocity gradient tensor.

#### Velocity Gradient Tensor

The velocity gradient tensor $\nabla \mathbf{u}$ has components:

$$
(\nabla \mathbf{u})_{ij} = \frac{\partial u_i}{\partial x_j}
$$

In 3D, this is a $3 \times 3$ tensor:

$$
\nabla \mathbf{u} = \begin{pmatrix}
\frac{\partial u}{\partial x} & \frac{\partial u}{\partial y} & \frac{\partial u}{\partial z} \\
\frac{\partial v}{\partial x} & \frac{\partial v}{\partial y} & \frac{\partial v}{\partial z} \\
\frac{\partial w}{\partial x} & \frac{\partial w}{\partial y} & \frac{\partial w}{\partial z}
\end{pmatrix}
$$

#### Decomposition into Strain and Rotation

The velocity gradient tensor can be decomposed into symmetric and antisymmetric parts:

$$
\nabla \mathbf{u} = \mathbf{S} + \mathbf{\Omega}
$$

**Strain Rate Tensor** (symmetric):
$$
\mathbf{S} = \frac{1}{2}\left(\nabla \mathbf{u} + (\nabla \mathbf{u})^T\right)
$$

**Rotation Rate Tensor** (antisymmetric):
$$
\mathbf{\Omega} = \frac{1}{2}\left(\nabla \mathbf{u} - (\nabla \mathbf{u})^T\right)
$$

#### Q-Criterion Definition

The Q-criterion is defined as:

$$
Q = \frac{1}{2}\left(\|\mathbf{\Omega}\|_F^2 - \|\mathbf{S}\|_F^2\right)
$$

where $\|\cdot\|_F$ denotes the Frobenius norm:

$$
\|\mathbf{A}\|_F^2 = \sum_{i,j} A_{ij}^2
$$

#### Physical Interpretation

- **$Q > 0$**: Rotation dominates strain → **vortex region**
- **$Q < 0$**: Strain dominates rotation → **strain-dominated region**
- **$Q = 0$**: Balance between rotation and strain

Vortices are visualized as **isosurfaces** of positive Q values. The threshold value determines which vortical structures are visible—higher thresholds show only the strongest vortices.

In [67]:
import numpy as np 
import pyvista as pv
import os
import argparse

pv.set_jupyter_backend("trame")
pv.global_theme.trame.jupyter_server_name = "my_viewer" 

def load_solution(filepath):
    print(f"Loading: {filepath}")
    data = np.load(filepath)
    return {
        'u': data['u'],
        'v': data['v'],
        'w': data['w'],
        'x': data['x'],
        'y': data['y'],
        'z': data['z'],
    }

def create_grid(data):
    """Convert numpy data to PyVista grid with velocity vector."""
    u, v, w = data['u'], data['v'], data['w']
    x, y, z = data['x'], data['y'], data['z']
    
    # Create PyVista Grid
    grid = pv.RectilinearGrid(x, y, z)
    
    # Transpose to (nx, ny, nz) and flatten F-order for PyVista
    u_t = u.transpose(2, 1, 0).flatten(order='F')
    v_t = v.transpose(2, 1, 0).flatten(order='F')
    w_t = w.transpose(2, 1, 0).flatten(order='F')
    
    # Stack into velocity vector
    velocity = np.column_stack((u_t, v_t, w_t))
    grid.point_data['velocity'] = velocity
    grid.point_data['u'] = u_t # Keep scalar u for basic isosurfaces
    
    return grid

def compute_q_criterion(grid):
    """
    Compute Q-criterion for vortex identification.
    Q = 0.5 * ( ||Omega||^2 - ||S||^2 )
    
    PyVista can compute derivatives (gradients) which we use to construct
    the velocity evaluation tensor.
    """
    print("Computing Q-criterion...")
    
    # PyVista's compute_derivative calculates the gradient of the vector field
    # Result is a (N, 9) array of gradients: du/dx, du/dy, du/dz, dv/dx...
    grid = grid.compute_derivative(scalars="velocity", gradient=True, faster=True)
    
    grad = grid["gradient"].reshape(-1, 3, 3) # (N, 3, 3)
    
    # Strain rate tensor S = 0.5 * (grad + grad.T)
    # Rotation tensor Omega = 0.5 * (grad - grad.T)
    
    grad_t = grad.transpose(0, 2, 1)
    S = 0.5 * (grad + grad_t)
    Omega = 0.5 * (grad - grad_t)
    
    # Frobenious norm squared
    norm_S2 = np.sum(S**2, axis=(1,2))
    norm_Omega2 = np.sum(Omega**2, axis=(1,2))
    
    Q = 0.5 * (norm_Omega2 - norm_S2)
    
    grid.point_data["Q"] = Q
    return grid

---

## Batch Visualization Function

The `visualize()` function provides a command-line interface for generating visualizations with different options:

| Argument | Description |
|----------|-------------|
| `--interactive` | Launch interactive 3D viewer (mouse rotation/zoom) |
| `--rotate` | Generate animated GIF with orbital rotation |
| `--q_criterion` | Visualize Q-criterion vortices (default: recirculation bubble) |
| `--iso_val` | Custom isosurface threshold value |

This function is useful for batch processing or script-based visualization.

In [68]:
def visualize(data, output_dir, args, h_step=0.4845):
    grid = create_grid(data)
    
    # Setup Plotter
    off_screen = not args.interactive
    pl = pv.Plotter(off_screen=off_screen)
    pl.set_background('white')
    
    # Add Step Geometry
    x, z = data['x'], data['z']
    step_box = pv.Box(bounds=[x.min(), 0, 0, h_step, z.min(), z.max()])
    pl.add_mesh(step_box, color='lightgrey', opacity=1.0, show_edges=False)
    pl.add_mesh(grid.outline(), color="black")
    
    if args.q_criterion:
        # Vortex Detection
        grid = compute_q_criterion(grid)
        
        # Q > 0 indicates vortex regions. Value depends on scale.
        # Try a small positive value relative to max Q
        q_max = grid.point_data["Q"].max()
        iso_val = args.iso_val if args.iso_val else 0.01 * q_max
        
        print(f"Generating Q={iso_val:.4f} isosurface (Max Q={q_max:.4f})...")
        isosurf = grid.contour(isosurfaces=[iso_val], scalars='Q')
        
        pl.add_mesh(isosurf, color='orange', opacity=0.8, 
                    smooth_shading=True, show_edges=False,
                    label=f'Q-Criterion (Vortices)')
        save_name = "isosurface_Q"
    else:
        # Recirculation Bubble (u=0)
        print("Generating u=0 isosurface...")
        isosurf = grid.contour(isosurfaces=[0.0], scalars='u')
        
        pl.add_mesh(isosurf, color='dodgerblue', opacity=0.6, 
                    smooth_shading=True, show_edges=False,
                    label='Recirculation (u=0)')
        save_name = "isosurface_u0"

    # Camera Setup
    pl.camera_position = 'iso'
    pl.camera.azimuth = -45
    pl.camera.elevation = 30
    pl.add_axes()

    if args.interactive:
        print("\nInteractive Mode: Use mouse to rotate/zoom. Press 'q' to exit.")
        pl.show()
    elif args.rotate:
        print("\nGenerating rotation animation...")
        outpath = os.path.join(output_dir, f"{save_name}_rotation.gif")
        path = pl.open_gif(outpath)
        
        # Orbit animation
        pl.camera_position = 'iso'
        pl.camera.azimuth = -45
        pl.camera.elevation = 30
        
        n_frames = 36
        for i in range(n_frames):
            pl.camera.azimuth += 360 / n_frames
            pl.write_frame()
            
        pl.close()
        print(f"Saved animation: {outpath}")
    else:
        outpath = os.path.join(output_dir, f"{save_name}.png")
        pl.screenshot(outpath, transparent_background=False)
        pl.close()
        print(f"Saved snapshot: {outpath}")

---

## Interactive Visualization

### Load Data and Compute Q-Criterion

Load the simulation data, create the PyVista grid, and compute the Q-criterion field for vortex identification.

In [69]:
input_path = "../numerical_fluid_dynamics/simulations/backward_facing_step_3d/outputs/solution_3d.npz"
data = load_solution(input_path)

grid = create_grid(data)
grid = compute_q_criterion(grid)

# Setup Plotter
pl = pv.Plotter()

Loading: ../numerical_fluid_dynamics/simulations/backward_facing_step_3d/outputs/solution_3d.npz
Computing Q-criterion...


### Building the Visualization

The visualization combines multiple elements:

1. **Step Geometry**: A solid box representing the backward-facing step obstacle
2. **Domain Outline**: Wireframe showing the computational domain boundaries
3. **Q-Criterion Isosurface**: Surface where $Q = Q_{threshold}$, revealing vortical structures
4. **Streamlines**: Particle trajectories showing the flow path

#### Streamline Integration

Streamlines trace the path a massless particle would follow in the velocity field. Starting from seed points, the particle position is integrated:

$$
\frac{d\mathbf{x}}{dt} = \mathbf{u}(\mathbf{x}, t)
$$

For steady flows, this gives **streamlines** (tangent to velocity everywhere). Key parameters:

| Parameter | Description |
|-----------|-------------|
| `source_center` | Center of the seed point sphere |
| `source_radius` | Radius of the seed sphere |
| `n_points` | Number of streamlines to generate |
| `max_time` | Maximum integration time (path length) |

**Seed locations** for different flow features:
- **Inlet flow**: $(-2.0, 0.6, 1.5)$ — captures the main flow entering the domain
- **Recirculation zone**: $(0.5, 0.25, 1.9)$ — captures reversed flow behind the step
- **Upper channel**: $(-2.0, 0.8, 1.9)$ — captures flow passing over the step

In [115]:
pl.clear() 
pl.set_background('white')

# Add Step Geometry
h_step = 0.4845
x, z = data['x'], data['z']
step_box = pv.Box(bounds=[x.min(), 0, 0, h_step, z.min(), z.max()])
pl.add_mesh(step_box, color='grey', opacity=1.0, show_edges=True)
pl.add_mesh(grid.outline(), color="black")

# Add Isosurface (Q-criterion)
q_max = grid.point_data["Q"].max()
iso_val = 0.9 * q_max
isosurf = grid.contour(isosurfaces=[iso_val], scalars='Q')
pl.add_mesh(isosurf, color="green", opacity=0.2, smooth_shading=True, label='vortices')

# Add Streamlines                                          
streamlines = grid.streamlines(                            
  vectors='velocity',                                    
  source_radius=0.1,          # radius of seed sphere    
  source_center=(0.2, 0.4, 3.8),  # starting point for seeds (adjust to your domain)                              
  n_points=200,                # number of seed points    
  max_time=14.0,              # max integration time     
)

streamlines2 = grid.streamlines(                            
  vectors='velocity',                                    
  source_radius=0.1,          # radius of seed sphere    
  source_center=(0.2, 0.5, 1.8),  # starting point for seeds (adjust to your domain)                              
  n_points=200,                # number of seed points    
  max_time=14.0,              # max integration time     
)

pl.add_mesh(streamlines.tube(radius=0.005),                
cmap='coolwarm', label='streamlines')
pl.add_mesh(streamlines2.tube(radius=0.005),                
cmap='coolwarm', label='streamlines2')


pl.add_axes()
pl.show()

/Users/pacho-home-server/numerical_fluid_dynamics/venv_vis/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:3204: PyVistaDeprecationWarning: ``max_time`` parameter is deprecated.  It will be removed in v0.48
  warnings.warn(


A view with name (P_0x1452ab390_3) is already registered
 => returning previous one


Widget(value='<iframe src="http://localhost:50559/index.html?ui=P_0x1452ab390_3&reconnect=auto" class="pyvista…

### Camera Control

After interactively positioning the view, retrieve the camera parameters for reproducible visualizations.

The camera position is a tuple of three vectors:
1. **Camera location**: $(x, y, z)$ position of the camera
2. **Focal point**: $(x, y, z)$ point the camera looks at
3. **View up vector**: Direction defining "up" in the view

In [116]:
print(pl.camera_position) 

[(9.34550993437214, 1.5567069169920054, -0.43872509418465866),
 (6.056700229644775, 0.4975668981205672, 1.940596103668213),
 (-0.12850070362723134, 0.9597798085181656, 0.24962028829432087)]


### Saving the Visualization

Export the current view as a high-resolution PNG image. The `window_size` parameter controls the output resolution.

In [117]:
pl.screenshot('my_flow.png', transparent_background=False, 
  window_size=(1920, 1080)) 

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[229, 216, 208],
                  [229, 216, 208],
                  [229, 216, 208],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[229, 216, 208],
  